## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Aturan Asosiasi Keranjang Belanja](images/img_10_market_basket.png)

```
        +--------------------------------------------------------------+
        |                 METRIK UTAMA ASOSIASI (A -> B)               |
        +--------------------------------------------------------------+
        |                                                              |
        |  [1] SUPPORT(A -> B) = P(A dan B)                            |
        |      Proporsi transaksi yang membeli item A dan B bersamaan. |
        |                                                              |
        |  [2] CONFIDENCE(A -> B) = P(B | A) = P(A dan B) / P(A)       |
        |      Probabilitas konsumen membeli B jika telah membeli A.   |
        |                                                              |
        |  [3] LIFT(A -> B) = P(A dan B) / [ P(A) * P(B) ]             |
        |      - Lift = 1.0 -> A dan B independen (kebetulan)          |
        |      - Lift > 1.0 -> Aturan positif kuat (Saling memicu)     |
        |      - Lift < 1.0 -> Hubungan substitutif / saling menolak   |
        +--------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_trx = pd.read_csv("../datasets/08_market_basket_transactions.csv")
print("Dataset Transaksi Keranjang Belanja dimuat. Total transaksi:", len(df_trx))
display(df_trx.head(8))


## 🛒 3. Transformasi Data Transaksi ke Bentuk One-Hot Encoded Basket


In [ ]:
# Mengurai string item menjadi list of items
transactions = df_trx['items'].apply(lambda x: [item.strip() for item in x.split(',')]).tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Dimensi Matriks Keranjang Belanja: {df_basket.shape[0]} Transaksi, {df_basket.shape[1]} Jenis Produk Unik")
print("\n5 Baris Pertama Matriks Biner Keranjang Belanja:")
display(df_basket.head())


## ⛏️ 4. Ekstraksi Frequent Itemsets & Aturan Asosiasi (Apriori)


In [ ]:
# Menemukan itemset yang sering muncul dengan min_support = 0.08 (8%)
frequent_itemsets = apriori(df_basket, min_support=0.08, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f"Total Frequent Itemsets Ditemukan: {len(frequent_itemsets)}")
display(frequent_itemsets.sort_values(by='support', ascending=False).head(10))

# Menghasilkan aturan asosiasi dengan min_threshold Lift = 1.2
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Memilih kolom penting
rules_display = rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage', 'conviction']]
rules_display = rules_display.sort_values(by='lift', ascending=False).reset_index(drop=True)

print("\n=== Top 10 Aturan Asosiasi Terbaik Berdasarkan Nilai Lift ===")
display(rules_display.head(10).round(3))


## 📊 5. Visualisasi Aturan Asosiasi: Scatter Support vs. Confidence dengan Hue Lift


In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', s=rules['lift']*50, alpha=0.8)
cbar = plt.colorbar(scatter)
cbar.set_label('Nilai Lift (Kekuatan Asosiasi)', rotation=270, labelpad=15)
plt.title('Evaluasi Aturan Asosiasi: Support vs. Confidence (Warna = Lift)', fontweight='bold')
plt.xlabel('Support (Frekuensi Bersama)')
plt.ylabel('Confidence (Tingkat Keyakinan)')
plt.grid(True)
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Mengapa nilai Lift lebih dipercaya daripada nilai Confidence saja?** Karena Confidence tidak memperhitungkan popularitas dasar produk konsekuen. Jika produk B sangat populer di seluruh toko, Confidence $A 
ightarrow B$ akan tampak tinggi secara semu. Nilai Lift $> 1.0$ membuktikan bahwa pembelian B benar-benar dipicu oleh pembelian A.

### Data Analysis Key Findings
* Aturan asosiasi dengan nilai Lift tertinggi mencakup pasangan produk komplementer:
  * $\{	ext{Laptop}\} 
ightarrow \{	ext{Wireless Mouse}, 	ext{Laptop Bag}\}$ ($	ext{Lift} > 3.0$)
  * $\{	ext{Smartphone}\} 
ightarrow \{	ext{Fast Charger}, 	ext{Screen Protector}\}$ ($	ext{Lift} > 2.8$)
* Nilai Confidence pada aturan-aturan kunci mencapai $>70\%$, menunjukkan reliabilitas pola transaksi yang tinggi.

### Insights or Next Steps
* Aturan asosiasi ini dapat langsung diterapkan untuk rekomendasi *Frequently Bought Together* dan tata letak *bundling* produk e-commerce.
